In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [2]:
#Загрузка данных
data_full = pd.concat([pd.read_csv('train.csv', index_col='id'), 
                       pd.read_csv('test.csv', index_col='id')])

In [3]:
#Категориальные признаки сбалансированы
for col in data_full.columns:
    if data_full[col].dtype == 'O':
        print(data_full[col].value_counts(), end=f'\n{'-'*40}\n')

gender
Female    426673
Male      422176
Name: count, dtype: int64
----------------------------------------
Partner
Yes    444257
No     404592
Name: count, dtype: int64
----------------------------------------
Dependents
No     591378
Yes    257471
Name: count, dtype: int64
----------------------------------------
PhoneService
Yes    797066
No      51783
Name: count, dtype: int64
----------------------------------------
MultipleLines
No                  403636
Yes                 393430
No phone service     51783
Name: count, dtype: int64
----------------------------------------
InternetService
Fiber optic    389117
DSL            258693
No             201039
Name: count, dtype: int64
----------------------------------------
OnlineSecurity
No                     412167
Yes                    235643
No internet service    201039
Name: count, dtype: int64
----------------------------------------
OnlineBackup
No                     355485
Yes                    292325
No internet service

In [4]:
#Создание dummy-переменных
data_dummy = pd.concat([pd.get_dummies(data_full.drop('Churn', axis=1)),
                        data_full['Churn']],
                       axis=1)

In [8]:
#Создание training и resulting
data_training = data_dummy[data_dummy['Churn'].notnull()]
data_resulting = data_dummy[data_dummy['Churn'].isnull()]

In [26]:
#Разделение на train и test
X = data_training.drop('Churn', axis=1)
y = data_training['Churn'].map({'No':0, 'Yes':1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

In [27]:
#Создание модели
xgb_model = XGBClassifier()
grid_params = {'n_estimators': [10,50],
               'max_depth': [1,3,5]}
grid_model = GridSearchCV(estimator=xgb_model,
                          param_grid=grid_params,
                          scoring='roc_auc',
                          cv=5,
                          verbose=2,
                          error_score='raise')

In [28]:
grid_model.fit(X_train, y_train)

Fitting 5 folds for each of 6 candidates, totalling 30 fits
[CV] END .......................max_depth=1, n_estimators=10; total time=   0.3s
[CV] END .......................max_depth=1, n_estimators=10; total time=   0.3s
[CV] END .......................max_depth=1, n_estimators=10; total time=   0.3s
[CV] END .......................max_depth=1, n_estimators=10; total time=   0.3s
[CV] END .......................max_depth=1, n_estimators=10; total time=   0.3s
[CV] END .......................max_depth=1, n_estimators=50; total time=   0.6s
[CV] END .......................max_depth=1, n_estimators=50; total time=   0.6s
[CV] END .......................max_depth=1, n_estimators=50; total time=   0.6s
[CV] END .......................max_depth=1, n_estimators=50; total time=   0.6s
[CV] END .......................max_depth=1, n_estimators=50; total time=   0.6s
[CV] END .......................max_depth=3, n_estimators=10; total time=   0.4s
[CV] END .......................max_depth=3, n_es

GridSearchCV(cv=5, error_score='raise',
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interac...
                                     learning_rate=None, max_bin=None,
                                     max_cat_threshold=None,
                                     max_cat_to_onehot=None,
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             param_grid={'max_depth': [1, 3, 5], 'n_estimators': [10, 50]},
             scoring='roc_auc', verbose=2)

In [30]:
y_predict = grid_model.predict(X_test)

In [31]:
print(classification_report(y_test, y_predict))

              precision    recall  f1-score   support

           0       0.90      0.92      0.91     92103
           1       0.71      0.64      0.67     26736

    accuracy                           0.86    118839
   macro avg       0.80      0.78      0.79    118839
weighted avg       0.86      0.86      0.86    118839



In [32]:
grid_model.predict_proba(X_test)

array([[0.46091247, 0.53908753],
       [0.05655229, 0.9434477 ],
       [0.56247354, 0.43752646],
       ...,
       [0.84793115, 0.15206882],
       [0.9750617 , 0.02493827],
       [0.68823314, 0.3117669 ]], dtype=float32)